# Vežba 4: Regresija u praksi i regularizacija

Metrike regresije, polinomska obeležja, underfitting i overfitting, Ridge i Lasso, krive učenja i vremenska podela

Lekcija sa objašnjenjima: [https://isatvss.surge.sh/vezba4.html](https://isatvss.surge.sh/vezba4.html)

Ćelije se izvršavaju redom, od vrha nadole (Shift+Enter), jer svaka koristi promenljive iz prethodnih.

## Metrike regresije

### Zadatak 1. Priprema podataka i vremenska podela

Učitati skup o vazduhu, dodati koncentraciju PM10 od prethodnog dana, mesec i dan u nedelji, pa podeliti podatke po vremenu: do kraja 2022. trening, 2023. i 2024. test.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

vazduh = pd.read_csv(
    "https://isatvss.surge.sh/podaci/vazduh_nis.csv", parse_dates=["datum"]
)
vazduh["pm10_juce"] = vazduh["pm10"].shift(1)
vazduh["mesec"] = vazduh["datum"].dt.month
vazduh["dan_u_nedelji"] = vazduh["datum"].dt.dayofweek
vazduh = vazduh.dropna(subset=["pm10", "pm10_juce"]).reset_index(drop=True)

trening = vazduh[vazduh["datum"].dt.year <= 2022]
test = vazduh[vazduh["datum"].dt.year >= 2023]
y_trening, y_test = trening["pm10"], test["pm10"]
print("Trening:", len(trening), "dana, test:", len(test), "dana")

### Zadatak 2. MAE, RMSE i R²

Istrenirati linearni model PM10 iz četiri meteorološke veličine, izračunati sve tri metrike ručno i funkcijama iz scikit-learn, i uporediti ih sa osnovnim modelom koji uvek predviđa prosek treninga.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

meteo = ["temperatura", "vetar_max", "padavine", "vlaznost"]
linearni = LinearRegression().fit(trening[meteo], y_trening)
predvidjeno = linearni.predict(test[meteo])
greske = predvidjeno - y_test

mae = np.mean(np.abs(greske))
rmse = np.sqrt(np.mean(greske**2))
r2 = 1 - np.sum(greske**2) / np.sum((y_test - y_test.mean()) ** 2)
print(f"Ručno:        MAE = {mae:.2f}, RMSE = {rmse:.2f}, R2 = {r2:.3f}")

mae_sk = mean_absolute_error(y_test, predvidjeno)
rmse_sk = np.sqrt(mean_squared_error(y_test, predvidjeno))
print(f"scikit-learn: MAE = {mae_sk:.2f}, RMSE = {rmse_sk:.2f}", end="")
print(f", R2 = {r2_score(y_test, predvidjeno):.3f}")

osnovni = np.full(len(y_test), y_trening.mean())
mae_osnovni = mean_absolute_error(y_test, osnovni)
r2_osnovni = r2_score(y_test, osnovni)
print(
    f"\nOsnovni model (prosek treninga): MAE = {mae_osnovni:.2f}, R2 = {r2_osnovni:.3f}"
)

## Underfitting i overfitting

### Zadatak 3. Polinomi stepena 1, 3 i 15

Na nasumičnom uzorku od 30 trening dana naučiti zavisnost PM10 od temperature polinomima stepena 1, 3 i 15, nacrtati krive i uporediti grešku na treningu i na testu.

Sa porastom stepena greška na treningu opada, od 23,35 do 20,12, jer složeniji polinom može bolje da prođe kroz 30 tačaka. Greška na testu se ponaša suprotno. Polinom 15. stepena između tačaka osciluje, a van opsega temperatura iz uzorka divlja, pa mu je RMSE na testu veći od četiri miliona. To je overfitting u najčistijem obliku: mala greška na treningu ne govori ništa o greški na novim podacima.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

uzorak = trening.sample(30, random_state=1)
x_uzorak, y_uzorak = uzorak[["temperatura"]], uzorak["pm10"]
x_crtanje = pd.DataFrame({"temperatura": np.linspace(-6, 30, 300)})

fig, ose = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for osa, stepen in zip(ose, [1, 3, 15]):
    model = make_pipeline(
        PolynomialFeatures(stepen), StandardScaler(), LinearRegression()
    )
    model.fit(x_uzorak, y_uzorak)
    rmse_tr = np.sqrt(mean_squared_error(y_uzorak, model.predict(x_uzorak)))
    rmse_te = np.sqrt(mean_squared_error(y_test, model.predict(test[["temperatura"]])))
    print(
        f"stepen {stepen:2d}: RMSE trening = {rmse_tr:6.2f}, RMSE test = {rmse_te:8.2f}"
    )
    osa.scatter(x_uzorak, y_uzorak, color="#d9480f", zorder=3, s=25)
    osa.plot(x_crtanje["temperatura"], model.predict(x_crtanje), color="#5b3cc4")
    osa.set_title(f"stepen {stepen}")
    osa.set_xlabel("Temperatura (°C)")
    osa.set_ylim(-20, 200)
ose[0].set_ylabel("PM10 (µg/m³)")
plt.tight_layout()
plt.show()

### Zadatak 4. Greška u zavisnosti od stepena

Za stepene od 1 do 15 izračunati RMSE na treningu i na testu i nacrtati obe krive.

Na ovih 30 dana najmanju grešku na testu ima prava, sa RMSE 22,98. Složeniji polinom bi imao smisla tek kada bi podataka bilo više, što se proverava u samostalnom zadatku 10. Stepen polinoma je hiperparametar i ne bira se po grešci na testu, nego validacijom, kao u zadatku 8. Ovde je test upotrebljen samo da bi se pojava pokazala.

In [ ]:
stepeni = range(1, 16)
rmse_trening, rmse_test = [], []
for stepen in stepeni:
    model = make_pipeline(
        PolynomialFeatures(stepen), StandardScaler(), LinearRegression()
    )
    model.fit(x_uzorak, y_uzorak)
    rmse_trening.append(np.sqrt(mean_squared_error(y_uzorak, model.predict(x_uzorak))))
    rmse_test.append(
        np.sqrt(mean_squared_error(y_test, model.predict(test[["temperatura"]])))
    )

plt.figure(figsize=(7, 4))
plt.plot(stepeni, rmse_trening, "-o", color="#5b3cc4", label="trening (30 dana)")
plt.plot(stepeni, rmse_test, "-o", color="#d9480f", label="test (2023-2024)")
plt.yscale("log")
plt.xlabel("Stepen polinoma")
plt.ylabel("RMSE (µg/m³), logaritamska skala")
plt.legend()
plt.show()

najbolji = int(np.argmin(rmse_test)) + 1
print(f"Najmanja greška na testu: stepen {najbolji}, RMSE = {min(rmse_test):.2f}")

## Regularizacija: Ridge i Lasso

### Zadatak 5. Ridge na polinomu 15. stepena

Na istih 30 dana naučiti polinom 15. stepena sa Ridge regularizacijom za tri vrednosti alpha i uporediti krive, grešku na testu i najveći koeficijent.

In [ ]:
from sklearn.linear_model import Ridge

fig, ose = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for osa, alfa in zip(ose, [0.001, 1, 100]):
    model = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=alfa))
    model.fit(x_uzorak, y_uzorak)
    rmse_te = np.sqrt(mean_squared_error(y_test, model.predict(test[["temperatura"]])))
    najveci = np.abs(model[-1].coef_).max()
    print(f"alpha = {alfa:7}: RMSE test = {rmse_te:6.2f}", end="")
    print(f", najveći |theta| = {najveci:7.2f}")
    osa.scatter(x_uzorak, y_uzorak, color="#d9480f", zorder=3, s=25)
    osa.plot(x_crtanje["temperatura"], model.predict(x_crtanje), color="#5b3cc4")
    osa.set_title(f"stepen 15, Ridge alpha = {alfa}")
    osa.set_xlabel("Temperatura (°C)")
    osa.set_ylim(-20, 200)
ose[0].set_ylabel("PM10 (µg/m³)")
plt.tight_layout()
plt.show()

### Zadatak 6. Lasso kao izbor obeležja

Na svim obeležjima i celom trening skupu naučiti Lasso za četiri vrednosti alpha, izbrojati koeficijente koji nisu nula i ispisati one koji ostaju pri najjačoj regularizaciji.

Pri alpha = 1 ostaje 5 od 24 koeficijenta, a greška na testu je gotovo ista kao sa svim obeležjima. Pri alpha = 3 ostaju samo temperatura, vetar i PM10 od juče. Ostala obeležja, uključujući mesec i dan u nedelji, ne donose mnogo kada su ova tri već poznata.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
from sklearn.preprocessing import OneHotEncoder

numericke = meteo + ["pm10_juce"]
kategoricke = ["mesec", "dan_u_nedelji"]
obelezja = numericke + kategoricke


def priprema():
    return ColumnTransformer(
        [
            ("numericke", StandardScaler(), numericke),
            ("kategoricke", OneHotEncoder(handle_unknown="ignore"), kategoricke),
        ]
    )


for alfa in [0.01, 0.1, 1, 3]:
    lasso = make_pipeline(priprema(), Lasso(alpha=alfa, max_iter=10000))
    lasso.fit(trening[obelezja], y_trening)
    koeficijenti = lasso[-1].coef_
    mae_te = mean_absolute_error(y_test, lasso.predict(test[obelezja]))
    print(
        f"alpha = {alfa:4}: nenultih koeficijenata {np.sum(koeficijenti != 0):2d}",
        end="",
    )
    print(f" od {len(koeficijenti)}, MAE test = {mae_te:.2f}")

nazivi = lasso[0].get_feature_names_out()
print("\nPreživeli koeficijenti za alpha = 3:")
print(pd.Series(koeficijenti, index=nazivi)[koeficijenti != 0].round(2))

## Krive učenja i konačni model

### Zadatak 7. Krive učenja

Učiti model na sve dužem početnom delu perioda 2019-2021, od 30 dana do svih, i svaki put izmeriti grešku na tim danima i na 2022. godini. Uporediti model sa meteorologijom i kalendarom i isti model uz PM10 od juče.

Kod modela bez jučerašnje vrednosti krive se spajaju: MAE je 14,52 na treningu i 14,55 na validaciji. Greška je ista na treningu i validaciji, ali visoka, što je slika underfitting-a, pa dodatni dani ne bi pomogli. Pomoglo je novo obeležje: uz PM10 od juče obe krive se spuštaju na oko 11. Model ni tada ne pokazuje znake overfitting-a.

In [ ]:
ucenje = trening[trening["datum"].dt.year <= 2021]
validacija = trening[trening["datum"].dt.year == 2022]
velicine = [30, 60, 120, 240, 480, len(ucenje)]


def model_sa(kolone):
    return make_pipeline(
        ColumnTransformer(
            [
                ("num", StandardScaler(), [k for k in kolone if k in numericke]),
                ("kat", OneHotEncoder(handle_unknown="ignore"), kategoricke),
            ]
        ),
        LinearRegression(),
    )


fig, ose = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
varijante = [
    ("meteorologija i kalendar", meteo + kategoricke),
    ("uz PM10 od juče", obelezja),
]
for osa, (naziv, kolone) in zip(ose, varijante):
    mae_tr, mae_val = [], []
    for n in velicine:
        deo = ucenje.iloc[:n]
        model = model_sa(kolone).fit(deo[kolone], deo["pm10"])
        mae_tr.append(mean_absolute_error(deo["pm10"], model.predict(deo[kolone])))
        mae_val.append(
            mean_absolute_error(validacija["pm10"], model.predict(validacija[kolone]))
        )
    osa.plot(velicine, mae_tr, "-o", color="#5b3cc4", label="trening")
    osa.plot(velicine, mae_val, "-o", color="#d9480f", label="validacija (2022)")
    osa.set_title(naziv)
    osa.set_xlabel("Broj dana za učenje")
    osa.set_ylim(0, 40)
    print(f"{naziv}: ", end="")
    print(f"MAE trening = {mae_tr[-1]:.2f}, MAE validacija = {mae_val[-1]:.2f}")
ose[0].set_ylabel("MAE (µg/m³)")
ose[0].legend()
plt.tight_layout()
plt.show()

### Zadatak 8. Izbor alpha i konačno poređenje

Izabrati alpha za Ridge unakrsnom validacijom koja poštuje vreme, pa na testu uporediti četiri modela: prosek treninga, „sutra kao danas“, linearni model sa meteorologijom i Ridge sa svim obeležjima.

Model „sutra kao danas“, koji za sutrašnji dan predviđa današnju vrednost, ima MAE 11,63 i jak je osnovni model za svaku vremensku seriju. Ridge sa svim obeležjima ga nadmašuje, sa MAE 10,41 i R² 0,648. Razlika nije velika, ali je pošteno izmerena.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

pretraga = GridSearchCV(
    make_pipeline(priprema(), Ridge()),
    {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    cv=TimeSeriesSplit(n_splits=5),
    scoring="neg_mean_absolute_error",
)
pretraga.fit(trening[obelezja], y_trening)
print("Izabrano alpha:", pretraga.best_params_["ridge__alpha"])

rezultati = {
    "prosek treninga": np.full(len(y_test), y_trening.mean()),
    "sutra kao danas": test["pm10_juce"],
    "linearni, meteorologija": linearni.predict(test[meteo]),
    "Ridge, sva obeležja": pretraga.predict(test[obelezja]),
}
for naziv, p in rezultati.items():
    mae_modela, r2_modela = mean_absolute_error(y_test, p), r2_score(y_test, p)
    print(f"{naziv:25s} MAE = {mae_modela:5.2f}, R2 = {r2_modela:6.3f}")

## Zadaci za samostalan rad

### Zadatak 9. Metrike za zadatak sa ispita

Za tačke i pravu h(x) = 2 + 0,5x sa vežbe 3 izračunati MAE, MSE, RMSE i R². Kako se MSE odnosi prema funkciji cene J izračunatoj na vežbi 3?

### Zadatak 10. Više podataka protiv overfitting-a

Naučiti polinom 15. stepena od temperature na 30, 300 i svim trening danima. Grešku na testu meriti samo za dane čija je temperatura u opsegu temperatura iz uzorka. Šta se dešava sa greškom kako podataka ima više?

### Zadatak 11. Putanja Lasso modela

Za sedam vrednosti alpha, od 0,01 do 10, izbrojati koeficijente Lasso modela koji nisu nula. Koje obeležje ostaje poslednje?

### Zadatak 12. Koliko vredi jučerašnja vrednost

Uporediti Ridge model (alpha = 10) sa svim obeležjima i bez koncentracije PM10 od juče. Zašto bi korišćenje današnje koncentracije PM2.5 za procenu današnjeg PM10 bilo pogrešno ako je cilj prognoza?

### Zadatak 13. Ridge i Lasso koeficijenti uporedo

Ispisati prvih osam koeficijenata Ridge (alpha = 10) i Lasso (alpha = 1) modela jedan pored drugog. Po čemu se razlikuju koeficijenti uz mesece?